In [2]:
!python -m pip install geopy


  Using cached geopy-2.4.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached geographiclib-2.0-py3-none-any.whl.metadata (1.4 kB)
Using cached geopy-2.4.1-py3-none-any.whl (125 kB)
Using cached geographiclib-2.0-py3-none-any.whl (40 kB)


In [3]:
import os
import pandas as pd
import googlemaps
from geopy.distance import geodesic


In [7]:
# kex
gmaps = googlemaps.Client(key=os.environ['kex_gmaps'])
# 5497
#gmaps = googlemaps.Client(key=os.environ['kushs5497_gmaps'])

pid_table = pd.read_csv('pid_table_2.csv')


In [8]:
api_call_count = 0

In [29]:
# Returns address string from street, city, county, state, zip
def create_address_string(street, city, county, state, zip):
    address = ''

    if street: address += f'{street}'
    if city: address += f', {city}'
    if county: address += f', {county}'
    if state: address += f', {state}'
    if zip: address += f', {zip}'

    return address

#
def get_pid_from_address(street, city, county, state, zip):
    # Search for address in pid_table
    if street and city and county and state and zip:
        filtered_pid = pid_table[(pid_table['I_Street'] == street) & 
                         (pid_table['I_City'] == city) & 
                         (pid_table['I_County'] == county) & 
                         (pid_table['I_State'] == state) & 
                         (pid_table['I_Zip'] == zip)]['PID']

        if not filtered_pid.empty:
            pid = filtered_pid.iloc[0]
            return pid
    
    address = create_address_string(street, city, county, state, zip)

    # Search for address in pid_table
    if address:
        filtered_pid = pid_table[pid_table['I_Address'] == address]['PID']

        if not filtered_pid.empty:
            pid = filtered_pid.iloc[0]
            return pid

    return None

def extract_address_components(response):
    # Define the components we want to extract
    desired_types = {
        "street_number": None,
        "route": None,
        "locality": None,
        "administrative_area_level_2": None,
        "administrative_area_level_1": None,
        "country": None,
        "postal_code": None,
        "postal_code_suffix": None
    }

    # Iterate through address components
    for component in response[0]['address_components']:
        for c_type in component['types']:
            if c_type in desired_types:
                desired_types[c_type] = component['long_name']

    return desired_types

def add_entry_to_pid_table(pid_table, pid, address, street, city, county, state, zip, lat, lng, formatted_address, street_number, route, locality, administrative_area_level_1, administrative_area_level_2, country, postal_code, postal_code_suffix):
    new_entry = {
        'PID': pid,
        'I_Address': address,
        'I_Street': street,
        'I_City': city,
        'I_County': county,
        'I_State': state,
        'I_Zip': zip,
        'G_Latitude': lat,
        'G_Longitude': lng,
        'G_Formatted Address': formatted_address,
        'G_Street Number': street_number,
        'G_Street': route,
        'G_City': locality,
        'G_State': administrative_area_level_1,
        'G_County': administrative_area_level_2,
        'G_Country': country,
        'G_Zip': postal_code,
        'G_Zip Suffix': postal_code_suffix
    }
    # Append the new row and return
    return pd.concat([pid_table, pd.DataFrame([new_entry])], ignore_index=True)


def geocode_address(street, city, county, state, zip):
    pid_in_table = get_pid_from_address(street, city, county, state, zip)
    if pid_in_table:
        return pid_in_table

    global api_call_count
    
    address = create_address_string(street, city, county, state, zip)
    
    geocode = gmaps.geocode(address)
    api_call_count += 1

    lat = geocode[0]['geometry']['location']['lat']
    lng = geocode[0]['geometry']['location']['lng']
    formatted_address = geocode[0]['formatted_address']

    address_components = extract_address_components(geocode)
    street_number = address_components['street_number']
    route = address_components['route']
    locality = address_components['locality']
    administrative_area_level_2 = address_components['administrative_area_level_2']
    administrative_area_level_1 = address_components['administrative_area_level_1']
    country = address_components['country']
    postal_code = address_components['postal_code']
    postal_code_suffix = address_components['postal_code_suffix']

    pid = geocode[0]['place_id']

    # Add these into pid_table
    pid_table = pd.concat([pid_table, pd.DataFrame({
        'PID': [pid],
        'I_Address': [address],
        'I_Street': [street],
        'I_City': [city],
        'I_County': [county],
        'I_State': [state],
        'I_Zip': [zip],
        'G_Latitude': [lat],
        'G_Longitude': [lng],
        'G_Formatted Address': [formatted_address],
        'G_Street Number': [street_number],
        'G_Street': [route],
        'G_City': [locality],
        'G_State': [administrative_area_level_1],
        'G_County': [administrative_area_level_2],
        'G_Country': [country],    
        'G_Zip': [postal_code],
        'G_Zip Suffix': [postal_code_suffix]
    })])


    if pid_table.shape[0] % 100 == 0:
        pid_table.to_csv('pid_table.csv', index=False)
        print(f'API Calls: {api_call_count}')
    
    return pid

def create_js_directories(df):
    saving_directory = 'Data_By_Towns_Index'
    assert df['County'].unique().shape[0] == 1
    county = df['County'].unique()[0]
    os.makedirs(f'{saving_directory}/{county}', exist_ok=True)

    for town in df['Municipality'].unique():
        #os.makedirs(f'Data_By_Towns_Index/{county}/{town}', exist_ok=True)
        town_df = df[df['Municipality'] == town]

        town_df = order_points_by_path(town_df)

        town_df['Notes'] = ''

        town_df.reset_index(drop=True, inplace=True)
        town_df.to_excel(f'{saving_directory}/{county}/{town}.xlsx', index=False)


# Helper function to compute path using nearest-neighbor heuristic
def order_points_by_path(df):
    points = df[['Latitude', 'Longitude']].values.tolist()
    ordered_indices = []
    remaining_points = points[:]
    current_index = 0

    while remaining_points:
        ordered_indices.append(current_index)
        current_point = points[current_index]
        remaining_points.remove(current_point)
        if not remaining_points:
            break
        # Find the nearest point
        current_index = points.index(min(remaining_points, key=lambda p: geodesic(current_point, p).meters))

    return df.iloc[ordered_indices].reset_index(drop=True)




In [30]:
tdh_directory = 'downloaded_addys'
wipp_directory = 'Data_By_Towns'

counties = ['Camden']

for county in counties:
    tdh_df = pd.read_excel(f'{tdh_directory}/{county}.xlsx')
    tdh_df['County'] = county
    tdh_df['State'] = 'NJ'
    wipp_df = pd.DataFrame()
    for town in os.listdir(wipp_directory+'/'+county):
        wipp_df = pd.concat([wipp_df, pd.read_csv(wipp_directory+'/'+county+'/'+town)])
    wipp_df['County'] = county
    wipp_df['State'] = 'NJ'

    tdh_df = tdh_df[['PamsPin', 'OwnerName', 'OwnerStreet', 'OwnerCityState', 'OwnerZipCode', 'County', 'State', 'PropertyLocation', 'PropertyClassCode']]
    # Apply geocode_address to the DataFrame
    tdh_df['OwnerPID'] = tdh_df.apply(lambda row: geocode_address(street=row['OwnerStreet'], 
                                                                  city=row['OwnerCityState'], 
                                                                  county=row['County'], 
                                                                  state=row['State'], 
                                                                  zip=row['OwnerZipCode']), axis=1)
    
    tdh_df['PropertyPID'] = tdh_df.apply(lambda row: geocode_address(street=row['PropertyLocation'], 
                                                                  city=row['OwnerCityState'], 
                                                                  county=row['County'], 
                                                                  state=row['State'], 
                                                                  zip=row['OwnerZipCode']), axis=1)
    
    wipp_df['PID'] = wipp_df.apply(lambda row: geocode_address(street=row['Property Location'], 
                                                               city=row['Municipality'], 
                                                               county=row['County'], 
                                                               state=row['State'], 
                                                               zip=None), axis=1)
    
    tdh_df['Rental'] = tdh_df['OwnerPID'] != tdh_df['PropertyPID']
    tdh_df['PID'] = tdh_df['OwnerPID']
    tdh_df.drop(columns=['OwnerPID', 'PropertyPID'], inplace=True)

    wipp_tdh_df = wipp_df.merge(tdh_df, how='left', on='PID')
    wipp_tdh_df = wipp_tdh_df[~(wipp_tdh_df['Rental'].fillna(False))]
    wipp_tdh_df = wipp_tdh_df[wipp_tdh_df['PropertyClassCode'] == '2']

    wipp_tdh_df.drop(columns=['Unnamed: 0', 'Search', 'County_y', 'State_y', 'PamsPin', 'OwnerName', 'OwnerStreet', 'OwnerCityState', 'OwnerZipCode', 'PropertyLocation', 'PropertyClassCode', 'Rental'], inplace=True)
    wipp_tdh_df.rename(columns={'County_x': 'County', 'State_x': 'State'}, inplace=True)
    wipp_tdh_df['Longitude'] = wipp_tdh_df['PID'].apply(lambda x: pid_table[pid_table['PID'] == x]['G_Longitude'].iloc[0])
    wipp_tdh_df['Latitude'] = wipp_tdh_df['PID'].apply(lambda x: pid_table[pid_table['PID'] == x]['G_Latitude'].iloc[0])

    wipp_tdh_df = wipp_tdh_df[['Owner Name', 'Property Location', 'Municipality', 'County', 'State', 'Longitude', 'Latitude', 'PID']].reset_index(drop=True)

    # wipp_tdh_df = order_points_by_path(wipp_tdh_df)


    create_js_directories(wipp_tdh_df)

    print(county)


print(api_call_count)
tdh_df
wipp_df
wipp_tdh_df

/Users/sunilpc/Desktop/VSCode/LB_Screener copy/venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/var/folders/bg/kvqrkgk13nd6gv909wz30xk80000gn/T/ipykernel_78017/2038933961.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  wipp_tdh_df = wipp_tdh_df[~(wipp_tdh_df['Rental'].fillna(False))]


Camden
2502


,Owner Name,Property Location,Municipality,County,State,Longitude,Latitude,PID
0,"PATEL, HASMUKHBH A. & URMILABEN H.",208 HAAKON RD,Brooklawn Borough,Camden,NJ,-75.119545,39.878889,ChIJSTqFWA_PxokRuKvQAALwDG4
1,DESAI AMOLA K & SHUBH K,3 SIDNEY LANE,Stratford Borough,Camden,NJ,-74.997715,39.829977,ChIJz0HO2m7NxokRhLGtJwCa8UU
2,"DESAI, AMOLA K & SHUBH K",3 SIDNEY LANE,Stratford Borough,Camden,NJ,-74.997715,39.829977,ChIJz0HO2m7NxokRhLGtJwCa8UU
3,MISTRY AMITKUMAR & SHEETAL JAGDISH,5 SIDNEY LANE,Stratford Borough,Camden,NJ,-74.999448,39.830855,ChIJWSekoAnNxokRXO272V5M8Vo
4,"MISTRY, AMITKUMAR N & SHEETAL JAGDI",5 SIDNEY LANE,Stratford Borough,Camden,NJ,-74.999448,39.830855,ChIJWSekoAnNxokRXO272V5M8Vo
...,...,...,...,...,...,...,...,...
690,SONI VISHAL & KALIA KSHMA,246 SANDRINGHAM RD,Cherry Hill Township,Camden,NJ,-74.928745,39.875376,ChIJ9aWXHHEzwYkRA54S2_wlvT8
691,"SONI, AJAY & SHANNA",12 HADLEIGH TERR,Cherry Hill Township,Camden,NJ,-74.943916,39.869726,ChIJJV9YURUzwYkRqqTBuEKZvyM
692,TRIVEDI RUSHIK,1102 SPRINGDALE RD,Cherry Hill Township,Camden,NJ,-74.970749,39.880226,ChIJV17UprPMxokR00-Z5LkzwO8
693,UPADHYAY ASHVIN B & NITA A & NAMAN,106 SIMI CT,Cherry Hill Township,Camden,NJ,-74.993076,39.862801,ChIJh4PIF-fMxokRP-Dznx2CspQ


In [27]:
pid_table.to_csv('pid_table_2.csv')
tdh_df.to_csv('tdh_df_2.csv')
wipp_df.to_csv('wipp_df_2.csv')

In [28]:
api_call_count

2502